In [8]:
import tensorflow as tf
from keras import layers, regularizers
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

print(f"TensorFlow version: {tf.__version__}")

TensorFlow version: 2.20.0


In [9]:
data = load_breast_cancer()
X, y = data.data, data.target
feature_names = data.feature_names
target_names = data.target_names

print(f"Dataset shape: {X.shape}")
print(f"Features: {len(feature_names)}")
print(f"Classes: {target_names}")
print(f"Class distribution: {np.bincount(y)}")

#normalizar features
scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
	X, y, test_size=0.2, random_state=42, stratify=y
)

#converter os arrays para tensores do tensorflow
X_train = tf.constant(X_train, dtype=tf.float32)
X_test = tf.constant(X_test, dtype=tf.float32)
y_train = tf.constant(y_train, dtype=tf.float32)
y_test = tf.constant(y_test, dtype=tf.float32)

#expandir dimensões do output para compatibilidade com modelo
y_train = tf.expand_dims(y_train, -1)
y_test = tf.expand_dims(y_test, -1)

print(f"\nTrain set: {X_train.shape}, {y_train.shape}")
print(f"Test set: {X_test.shape}, {y_test.shape}")


Dataset shape: (569, 30)
Features: 30
Classes: ['malignant' 'benign']
Class distribution: [212 357]

Train set: (455, 30), (455, 1)
Test set: (114, 30), (114, 1)


In [10]:
def create_model(model_type, n_features, hidden_units=64, dropout_rate=0.5, l1_reg=0.001, l2_reg=0.001 ):
	model = tf.keras.Sequential()
	model.add(layers.Input(shape=(n_features,)))

	if model_type == 'baseline':
		#modelo baseline sem regularização
		model.add(layers.Dense(hidden_units, activation='relu'))
		model.add(layers.Dense(hidden_units, activation='relu'))
		model.add(layers.Dense(hidden_units, activation='relu'))

	elif model_type == 'l1':
		#modelo com regularização L1
		model.add(layers.Dense(hidden_units, activation='relu',
							  kernel_regularizer=regularizers.l1(l1_reg)))
		model.add(layers.Dense(hidden_units, activation='relu',
							  kernel_regularizer=regularizers.l1(l1_reg)))
		model.add(layers.Dense(hidden_units, activation='relu',
							  kernel_regularizer=regularizers.l1(l1_reg)))

	elif model_type == 'l2':
		#modelo com regularização L2
		model.add(layers.Dense(hidden_units, activation='relu',
							  kernel_regularizer=regularizers.l2(l2_reg)))
		model.add(layers.Dense(hidden_units, activation='relu',
							  kernel_regularizer=regularizers.l2(l2_reg)))
		model.add(layers.Dense(hidden_units, activation='relu',
							  kernel_regularizer=regularizers.l2(l2_reg)))

	elif model_type == 'dropout':
		#modelo com dropout
		model.add(layers.Dense(hidden_units, activation='relu',))
		model.add(layers.Dropout(dropout_rate))
		model.add(layers.Dense(hidden_units, activation='relu'))
		model.add(layers.Dropout(dropout_rate))
		model.add(layers.Dense(hidden_units, activation='relu'))
		model.add(layers.Dropout(dropout_rate))

	elif model_type == 'combined':
		model.add(layers.Dense(hidden_units, kernel_regularizer=regularizers.l2(l2_reg),
		activation='relu'))
		model.add(layers.Dropout(dropout_rate))
		model.add(layers.Dense(hidden_units, kernel_regularizer=regularizers.l2(l2_reg),
		activation='relu'))
		model.add(layers.Dropout(dropout_rate))
		model.add(layers.Dense(hidden_units, kernel_regularizer=regularizers.l2(l2_reg),
		activation='relu'))
		model.add(layers.Dropout(dropout_rate))

	#camada de saída para classificação binária
	model.add(layers.Dense(1, activation='sigmoid'))

	return model

In [11]:
def train_model(model, X_train, y_train, X_test, y_test, epochs=100,batch_size=32, verbose=0):
		optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
		model.compile(
			optimizer=optimizer,
			loss=tf.keras.losses.BinaryCrossentropy(from_logits=False),  # Changed to False since we use sigmoid
			metrics=['accuracy']
		)

		callbacks = [
			tf.keras.callbacks.EarlyStopping(
				monitor='val_loss',
				patience=15,
				restore_best_weights=True
			),
			tf.keras.callbacks.ReduceLROnPlateau(
				monitor='val_loss',
				factor=0.5,
				patience=5,
				min_lr=1e-7
			)
		]

		history = model.fit(
			X_train, y_train,
			validation_data=(X_test, y_test),
			epochs=epochs,
			batch_size=batch_size,
			callbacks=callbacks,
			verbose=verbose
		)

		return history

In [12]:
model_types = ['baseline', 'l1', 'l2', 'dropout', 'combined']
results = {}
print("=" * 60)

for model_type in model_types:
	print(f"Treinando modelo {model_type}...")

	model = create_model(
		model_type,
		n_features=X_train.shape[1],
		hidden_units=64,
		dropout_rate=0.5,
		l1_reg=0.001,
		l2_reg=0.001
	)

	history = train_model(
		model, X_train, y_train, X_test, y_test,
		epochs=100, verbose=0
	)

	#avaliar modelo
	train_loss, train_acc = model.evaluate(X_train, y_train, verbose=0)
	test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)

	#salvar resultados
	results[model_type] = {
		'train_accuracy': train_acc,
		'test_accuracy': test_acc,
		'train_loss': train_loss,
		'test_loss': test_loss,
		'history': history,
		'model': model
	}

	print(f"  Train Acc: {train_acc:.4f}, Test Acc: {test_acc:.4f}")
	print(f"  Train Loss: {train_loss:.4f}, Test Loss: {test_loss:.4f}")
	print(f"  Overfitting: {train_acc - test_acc:.4f}")

print("=" * 60)







Treinando modelo baseline...
  Train Acc: 0.9890, Test Acc: 0.9561
  Train Loss: 0.0461, Test Loss: 0.0853
  Overfitting: 0.0329
Treinando modelo l1...
  Train Acc: 0.9956, Test Acc: 0.9561
  Train Loss: 0.0884, Test Loss: 0.1731
  Overfitting: 0.0395
Treinando modelo l2...
  Train Acc: 1.0000, Test Acc: 0.9561
  Train Loss: 0.0316, Test Loss: 0.1485
  Overfitting: 0.0439
Treinando modelo dropout...
  Train Acc: 0.9802, Test Acc: 0.9649
  Train Loss: 0.0678, Test Loss: 0.0941
  Overfitting: 0.0153
Treinando modelo combined...
  Train Acc: 0.9912, Test Acc: 0.9649
  Train Loss: 0.1293, Test Loss: 0.2022
  Overfitting: 0.0263


para o modelo de tamanho medio, o modelo com dropout e a combinação de dropout e regularização l2 apresentaram os melhores resultados.